# PIDA-RL Diabetes - 05. Politicas base autocontenidas

Este notebook corrige el problema de archivos temporales de Colab. En lugar de requerir que exista un archivo de entorno creado por un notebook anterior, define e importa una version verificable del entorno desde este mismo notebook.

> La cohorte permanece como unico insumo persistente obligatorio. Las transiciones y las recompensas son supuestos del simulador.


## Diagnostico del error anterior

`/content/pida-rl-datos-publicos` es temporal. Si Colab reinicia, se vuelve a clonar el repositorio y desaparecen los archivos creados manualmente durante una sesion anterior, incluido `src/environment/diabetes_followup_env_clean.py`. La cohorte persiste en Drive; el codigo creado dentro de `/content` no.


## 1. Inicializacion


In [1]:
from pathlib import Path
import os
import sys
import subprocess
import importlib
import json

PROJECT_NAME = 'pida-rl-datos-publicos'
REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
PROJECT_ROOT = Path('/content') / PROJECT_NAME

DRIVE_ROOT = next(
    (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
    None
)

if DRIVE_ROOT is None:
    from google.colab import drive
    mount_point = Path('/content/drive')
    if mount_point.exists() and any(mount_point.iterdir()):
        mount_point = Path('/content/gdrive')
    if mount_point.exists() and any(mount_point.iterdir()):
        raise RuntimeError('No hay un punto de montaje vacio para Google Drive. Reinicia la sesion.')
    drive.mount(str(mount_point), force_remount=False)
    DRIVE_ROOT = mount_point

MYDRIVE_ROOT = DRIVE_ROOT / 'MyDrive'
PERSISTENT_ROOT = MYDRIVE_ROOT / 'PIDA-RL-Diabetes'
DATA_ROOT = PERSISTENT_ROOT / 'data'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
TABLES_DIR = PERSISTENT_ROOT / 'results' / 'tables'
MODELS_DIR = PERSISTENT_ROOT / 'models'
LOGS_DIR = PERSISTENT_ROOT / 'logs'

for directory in [TABLES_DIR, MODELS_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])
elif not (PROJECT_ROOT / '.git').is_dir():
    raise RuntimeError(f'La ruta no es un repositorio Git: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
SRC_ROOT = PROJECT_ROOT / 'src'
SRC_ROOT.mkdir(parents=True, exist_ok=True)
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'
if not COHORT_PATH.is_file():
    raise FileNotFoundError(f'No existe la cohorte: {COHORT_PATH}. Ejecuta primero el notebook 03.')

print('Proyecto temporal:', PROJECT_ROOT)
print('Cohorte persistente:', COHORT_PATH)
print('Resultados persistentes:', TABLES_DIR)


Mounted at /content/drive
Proyecto temporal: /content/pida-rl-datos-publicos
Cohorte persistente: /content/drive/MyDrive/PIDA-RL-Diabetes/data/processed/cohorte_diabetes.csv
Resultados persistentes: /content/drive/MyDrive/PIDA-RL-Diabetes/results/tables


## 2. Cargar y validar cohorte


In [2]:
import numpy as np
import pandas as pd

df_cohorte_diabetes = pd.read_csv(COHORT_PATH, dtype={'FOLIO_INT': 'string'})

required_columns = [
    'FOLIO_INT', 'edad', 'sexo', 'anios_con_diabetes',
    'insulina_diaria', 'depresion', 'consultas_control_12m',
    'num_hospitalizaciones', 'num_complicaciones',
    'score_riesgo', 'categoria_riesgo'
]

missing = [c for c in required_columns if c not in df_cohorte_diabetes.columns]
if missing:
    raise KeyError('Faltan columnas en cohorte: ' + ', '.join(missing))

numeric_columns = [
    'edad', 'sexo', 'anios_con_diabetes', 'insulina_diaria',
    'depresion', 'consultas_control_12m', 'num_hospitalizaciones',
    'num_complicaciones', 'score_riesgo'
]

assert np.isfinite(df_cohorte_diabetes[numeric_columns].to_numpy(dtype=float)).all()
assert df_cohorte_diabetes['categoria_riesgo'].isin(['bajo', 'medio', 'alto']).all()
assert not df_cohorte_diabetes['FOLIO_INT'].duplicated().any()

print('Cohorte valida:', df_cohorte_diabetes.shape)
display(df_cohorte_diabetes.head())


Cohorte valida: (1542, 16)


,FOLIO_INT,edad,sexo,anios_con_diabetes,insulina_diaria,depresion,consultas_control_12m,num_hospitalizaciones,complicacion_pie,complicacion_visual,complicacion_renal,complicacion_macrovascular,evento_agudo_diabetes,num_complicaciones,score_riesgo,categoria_riesgo
0,2022_01001011_01,66.0,1.0,0.0,0,0,0.0,0.0,0,0,0,0,0,0,1,bajo
1,2022_01001015_01,65.0,2.0,20.0,1,1,0.0,0.0,0,1,0,0,0,1,6,alto
2,2022_01001017_02,70.0,2.0,8.0,0,1,0.0,0.0,0,1,0,0,0,1,4,alto
3,2022_01001018_02,65.0,1.0,40.0,0,0,0.0,0.0,0,1,1,0,0,2,6,alto
4,2022_01001027_01,76.0,1.0,56.0,0,0,0.0,0.0,0,0,0,0,0,0,2,medio


## 3. Definir el entorno autocontenido

El entorno se define aqui para que el notebook funcione incluso si el repositorio fue reclonado o si los archivos temporales de una sesion anterior ya no existen.


In [3]:
try:
    import gymnasium as gym
    from gymnasium import spaces
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'gymnasium'])
    import gymnasium as gym
    from gymnasium import spaces

class DiabetesFollowUpEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self, cohorte_df, capacidad_mensual=0.20, horizonte=12, seed=None):
        super().__init__()
        required = [
            'edad', 'sexo', 'anios_con_diabetes', 'insulina_diaria',
            'depresion', 'consultas_control_12m', 'num_hospitalizaciones',
            'num_complicaciones', 'score_riesgo', 'categoria_riesgo'
        ]
        missing = [name for name in required if name not in cohorte_df.columns]
        if missing:
            raise KeyError('Faltan columnas: ' + ', '.join(missing))
        if not 0 < float(capacidad_mensual) <= 1:
            raise ValueError('capacidad_mensual debe estar entre 0 y 1.')
        if int(horizonte) < 1:
            raise ValueError('horizonte debe ser al menos 1.')
        self.cohorte_base = cohorte_df.reset_index(drop=True).copy()
        self.n_perfiles = len(self.cohorte_base)
        if self.n_perfiles == 0:
            raise ValueError('La cohorte no puede estar vacia.')
        numeric = [
            'edad', 'sexo', 'anios_con_diabetes', 'insulina_diaria',
            'depresion', 'consultas_control_12m', 'num_hospitalizaciones',
            'num_complicaciones', 'score_riesgo'
        ]
        if not np.isfinite(self.cohorte_base[numeric].to_numpy(dtype=float)).all():
            raise ValueError('La cohorte contiene NaN o infinitos.')
        if not self.cohorte_base['categoria_riesgo'].isin({'bajo', 'medio', 'alto'}).all():
            raise ValueError('categoria_riesgo invalida.')
        self.capacidad_mensual = float(capacidad_mensual)
        self.horizonte = int(horizonte)
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(13,), dtype=np.float32)
        self.costo_accion = {0: 0, 1: 1, 2: 2, 3: 3}
        self.reduccion_evento = {0: 0.00, 1: 0.20, 2: 0.45, 3: 0.65}
        self.riesgo_ordinal = {'bajo': 0, 'medio': 1, 'alto': 2}
        self.reset(seed=seed)

    @staticmethod
    def _norm(value, denominator):
        return float(np.clip(float(value) / max(float(denominator), 1.0), 0.0, 1.0))

    def _risk_level(self, category):
        return self.riesgo_ordinal[str(category)]

    def _get_obs(self, idx):
        row = self.cohorte.iloc[idx]
        remaining = self.presupuesto_mensual - self.recursos_usados_mes
        sex_norm = 0.0 if float(row['sexo']) == 1.0 else 1.0
        risk_norm = self._risk_level(row['categoria_riesgo']) / 2.0
        return np.array([
            self._norm(row['edad'], 100),
            sex_norm,
            self._norm(row['anios_con_diabetes'], 80),
            float(np.clip(row['insulina_diaria'], 0, 1)),
            float(np.clip(row['depresion'], 0, 1)),
            self._norm(row['consultas_control_12m'], 24),
            self._norm(row['num_hospitalizaciones'], 12),
            self._norm(row['num_complicaciones'], 5),
            self._norm(row['score_riesgo'], 16),
            risk_norm,
            self._norm(row['meses_sin_contacto'], self.horizonte),
            self._norm(self.mes_actual, self.horizonte),
            self._norm(remaining, self.presupuesto_mensual)
        ], dtype=np.float32)

    def _event_probability(self, row, risk_level):
        value = (
            0.01 + 0.025 * risk_level
            + 0.010 * float(row['num_complicaciones'])
            + 0.020 * float(row['num_hospitalizaciones'] > 0)
            + 0.005 * min(float(row['meses_sin_contacto']), 6)
        )
        return float(np.clip(value, 0.0, 0.80))

    def _reward(self, action, risk_level, action_accepted, adverse_event, over_capacity):
        reward = 0.0
        if action_accepted and action > 0:
            reward += {1: 0.5, 2: 1.0, 3: 1.5}[action] * (risk_level + 1)
        if action == 0 or not action_accepted:
            reward -= {0: 0.0, 1: 0.5, 2: 1.5}[risk_level]
        if adverse_event:
            reward -= 5.0
        if over_capacity:
            reward -= 3.0
        return float(reward)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.cohorte = self.cohorte_base.copy()
        self.cohorte['meses_sin_contacto'] = 0
        self.mes_actual = 1
        self.idx_actual = 0
        self.presupuesto_mensual = max(1, int(round(self.n_perfiles * self.capacidad_mensual)))
        self.recursos_usados_mes = 0
        self.contactos_aceptados_mes = 0
        self.intentos_exceso_mes = 0
        obs = self._get_obs(self.idx_actual)
        info = {
            'mes': self.mes_actual,
            'indice_perfil': self.idx_actual,
            'n_perfiles': self.n_perfiles,
            'presupuesto_mensual': self.presupuesto_mensual,
            'recursos_usados_mes': 0,
            'recursos_restantes_mes': self.presupuesto_mensual
        }
        return obs, info

    def step(self, action):
        action = int(action)
        if not self.action_space.contains(action):
            raise ValueError(f'Accion invalida: {action}')
        idx = self.idx_actual
        month = self.mes_actual
        row = self.cohorte.iloc[idx]
        risk_level = self._risk_level(row['categoria_riesgo'])
        requested_cost = self.costo_accion[action]
        over_capacity = self.recursos_usados_mes + requested_cost > self.presupuesto_mensual
        action_accepted = not over_capacity
        actual_cost = requested_cost if action_accepted else 0
        if over_capacity:
            self.intentos_exceso_mes += 1
        else:
            self.recursos_usados_mes += actual_cost
            if action > 0:
                self.contactos_aceptados_mes += 1
        base_probability = self._event_probability(row, risk_level)
        reduction = self.reduccion_evento[action] if action_accepted else 0.0
        probability = float(np.clip(base_probability * (1.0 - reduction), 0.0, 1.0))
        adverse_event = bool(self.np_random.random() < probability)
        reward = self._reward(action, risk_level, action_accepted, adverse_event, over_capacity)
        if action > 0 and action_accepted:
            self.cohorte.loc[idx, 'meses_sin_contacto'] = 0
        else:
            self.cohorte.loc[idx, 'meses_sin_contacto'] += 1
        used = int(self.recursos_usados_mes)
        remaining = int(self.presupuesto_mensual - self.recursos_usados_mes)
        contacts = int(self.contactos_aceptados_mes)
        excess_attempts = int(self.intentos_exceso_mes)
        self.idx_actual += 1
        terminated = False
        truncated = False
        month_changed = False
        if self.idx_actual >= self.n_perfiles:
            self.idx_actual = 0
            self.mes_actual += 1
            month_changed = True
            self.recursos_usados_mes = 0
            self.contactos_aceptados_mes = 0
            self.intentos_exceso_mes = 0
            if self.mes_actual > self.horizonte:
                terminated = True
        obs = np.zeros(self.observation_space.shape, dtype=np.float32) if terminated else self._get_obs(self.idx_actual)
        info = {
            'mes': month,
            'mes_siguiente': self.mes_actual,
            'cambio_mes': month_changed,
            'indice_perfil': idx,
            'nivel_riesgo': risk_level,
            'categoria_riesgo': str(row['categoria_riesgo']),
            'evento_adverso': adverse_event,
            'prob_evento': probability,
            'excede_capacidad': over_capacity,
            'accion_aceptada': action_accepted,
            'costo_solicitado': requested_cost,
            'costo_real': actual_cost,
            'presupuesto_mensual': self.presupuesto_mensual,
            'recursos_usados_mes': used,
            'recursos_restantes_mes': remaining,
            'contactos_aceptados_mes': contacts,
            'intentos_exceso_mes': excess_attempts
        }
        return obs, reward, terminated, truncated, info

print('Clase DiabetesFollowUpEnv definida dentro del notebook.')


Clase DiabetesFollowUpEnv definida dentro del notebook.


## 4. Validar entorno actual


In [4]:
try:
    import stable_baselines3
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'stable-baselines3'])

from stable_baselines3.common.env_checker import check_env

env_check = DiabetesFollowUpEnv(df_cohorte_diabetes, capacidad_mensual=0.20, horizonte=12, seed=42)
check_env(env_check, warn=True)

obs, info = env_check.reset(seed=42)
assert obs.shape == (13,)
assert env_check.observation_space.contains(obs)
assert np.isfinite(obs).all()

print('Entorno autocontenido validado.')
print('Observacion:', env_check.observation_space)
print('Acciones:', env_check.action_space)
print('Presupuesto mensual:', env_check.presupuesto_mensual)


Entorno autocontenido validado.
Observacion: Box(0.0, 1.0, (13,), float32)
Acciones: Discrete(4)
Presupuesto mensual: 308


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 5. Politicas de referencia


In [5]:
def politica_aleatoria(env, rng):
    return int(rng.integers(0, env.action_space.n))

def politica_reglas_riesgo(env, rng):
    categoria = str(env.cohorte.iloc[env.idx_actual]['categoria_riesgo'])
    if categoria == 'alto':
        return 3
    if categoria == 'medio':
        return 2
    return 1 if rng.random() < 0.10 else 0

def politica_reglas_capacidad(env, rng):
    categoria = str(env.cohorte.iloc[env.idx_actual]['categoria_riesgo'])
    restante = env.presupuesto_mensual - env.recursos_usados_mes
    accion = {'alto': 3, 'medio': 2, 'bajo': 1}[categoria]
    if categoria == 'bajo' and rng.random() >= 0.10:
        accion = 0
    if env.costo_accion[accion] <= restante:
        return int(accion)
    factibles = [a for a, costo in env.costo_accion.items() if costo <= restante]
    positivas = [a for a in factibles if a > 0]
    return int(max(positivas)) if positivas else 0

print('Politicas base definidas.')


Politicas base definidas.


## 6. Evaluador comun


In [6]:
def evaluate_policy(env_factory, policy_fn, policy_name, seeds=(42, 123, 2026)):
    summary_rows = []
    step_rows = []

    for seed in seeds:
        env = env_factory(seed)
        rng = np.random.default_rng(seed)
        obs, _ = env.reset(seed=seed)
        total_reward = 0.0
        episode_rows = []
        step_number = 0

        while True:
            action = int(policy_fn(env, obs, rng))
            next_obs, reward, terminated, truncated, info = env.step(action)
            step_number += 1
            total_reward += float(reward)

            row = {
                'politica': policy_name,
                'semilla': int(seed),
                'paso': int(step_number),
                'mes': int(info['mes']),
                'nivel_riesgo': int(info['nivel_riesgo']),
                'categoria_riesgo': info['categoria_riesgo'],
                'accion': action,
                'accion_aceptada': bool(info['accion_aceptada']),
                'contacto_aceptado': bool(action > 0 and info['accion_aceptada']),
                'evento_adverso': bool(info['evento_adverso']),
                'recompensa': float(reward),
                'costo_real': int(info['costo_real']),
                'excede_capacidad': bool(info['excede_capacidad'])
            }

            episode_rows.append(row)
            step_rows.append(row)
            obs = next_obs

            if terminated or truncated:
                break

        episode = pd.DataFrame(episode_rows)
        grouped = episode.groupby(['nivel_riesgo', 'categoria_riesgo'], as_index=False).agg(
            decisiones=('accion', 'size'),
            contactos_aceptados=('contacto_aceptado', 'sum'),
            eventos_adversos=('evento_adverso', 'sum'),
            recursos_usados=('costo_real', 'sum'),
            excesos_capacidad=('excede_capacidad', 'sum')
        )

        grouped['cobertura_porcentaje'] = np.where(
            grouped['decisiones'] > 0,
            100.0 * grouped['contactos_aceptados'] / grouped['decisiones'],
            0.0
        )

        grouped['politica'] = policy_name
        grouped['semilla'] = int(seed)
        grouped['recompensa_acumulada_episodio'] = float(total_reward)
        grouped['pasos_episodio'] = int(step_number)
        summary_rows.extend(grouped.to_dict(orient='records'))

    return pd.DataFrame(summary_rows), pd.DataFrame(step_rows)

def env_factory(seed):
    return DiabetesFollowUpEnv(
        cohorte_df=df_cohorte_diabetes,
        capacidad_mensual=0.20,
        horizonte=12,
        seed=seed
    )

def wrap_policy(policy):
    def wrapped(env, obs, rng):
        return policy(env, rng)
    return wrapped

print('Evaluador listo.')


Evaluador listo.


## 7. Evaluar politicas base


In [7]:
SEEDS = (42, 123, 2026)

policy_specs = [
    ('aleatoria', wrap_policy(politica_aleatoria)),
    ('reglas_riesgo', wrap_policy(politica_reglas_riesgo)),
    ('reglas_capacidad', wrap_policy(politica_reglas_capacidad))
]

summary_parts = []
step_parts = []

for policy_name, policy_fn in policy_specs:
    summary, steps = evaluate_policy(env_factory, policy_fn, policy_name, seeds=SEEDS)
    summary_parts.append(summary)
    step_parts.append(steps)
    print(policy_name, 'pasos:', len(steps))

df_baselines_summary = pd.concat(summary_parts, ignore_index=True)
df_baselines_steps = pd.concat(step_parts, ignore_index=True)

comparison = (
    df_baselines_summary
    .groupby('politica', as_index=False)
    .agg(
        recompensa_media=('recompensa_acumulada_episodio', 'mean'),
        recompensa_desviacion=('recompensa_acumulada_episodio', 'std'),
        cobertura_media=('cobertura_porcentaje', 'mean'),
        contactos_medios=('contactos_aceptados', 'mean'),
        eventos_adversos_medios=('eventos_adversos', 'mean'),
        recursos_medios=('recursos_usados', 'mean'),
        excesos_medios=('excesos_capacidad', 'mean')
    )
)

high_risk = (
    df_baselines_summary
    .query('categoria_riesgo == "alto"')
    .groupby('politica', as_index=False)
    .agg(cobertura_alto_riesgo=('cobertura_porcentaje', 'mean'))
)

comparison = comparison.merge(high_risk, on='politica', how='left').round(2)
display(comparison.sort_values('recompensa_media', ascending=False))


aleatoria pasos: 55512


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

reglas_riesgo pasos: 55512


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

reglas_capacidad pasos: 55512


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,politica,recompensa_media,recompensa_desviacion,cobertura_media,contactos_medios,eventos_adversos_medios,recursos_medios,excesos_medios,cobertura_alto_riesgo
1,reglas_capacidad,-13521.00,123.25,6.97,480.44,384.78,1232.0,0.00,11.76
0,aleatoria,-50511.33,124.49,9.93,617.56,384.44,1232.0,4004.00,10.94
2,reglas_riesgo,-52178.67,156.03,6.98,480.44,384.78,1232.0,4294.78,11.76


## 8. Guardar resultados persistentes


In [8]:
summary_path = TABLES_DIR / 'evaluacion_politicas_base_v1.csv'
steps_path = TABLES_DIR / 'evaluacion_pasos_politicas_base_v1.csv'
comparison_path = TABLES_DIR / 'comparacion_politicas_base_v1.csv'

df_baselines_summary.to_csv(summary_path, index=False, encoding='utf-8-sig')
df_baselines_steps.to_csv(steps_path, index=False, encoding='utf-8-sig')
comparison.to_csv(comparison_path, index=False, encoding='utf-8-sig')

for path in [summary_path, steps_path, comparison_path]:
    print(path.name, '| existe:', path.is_file(), '| bytes:', path.stat().st_size if path.is_file() else 0)

assert np.isfinite(df_baselines_steps['recompensa']).all()
assert (df_baselines_steps['costo_real'] >= 0).all()
print('Politicas base evaluadas y guardadas correctamente.')


evaluacion_politicas_base_v1.csv | existe: True | bytes: 2345
evaluacion_pasos_politicas_base_v1.csv | existe: True | bytes: 10702614
comparacion_politicas_base_v1.csv | existe: True | bytes: 367
Politicas base evaluadas y guardadas correctamente.


## Resultado esperado

El notebook debe terminar con `Politicas base evaluadas y guardadas correctamente.` y tres archivos CSV en `results/tables`. El siguiente notebook debe entrenar DQN usando el mismo entorno definido aqui o, preferentemente, un modulo versionado y persistido en el repositorio Git.
